# FDS Challenge: My Notebook

## 0. Configuration

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


DEBUG = False

pd.set_option("display.max_columns", None)  # keep it! it's usefull to make pandas print also the null columns
pd.set_option("display.width", None)


# --- PATHS ---
TRAIN_FILE_PATH        = "../dataset/train.jsonl"
TEST_FILE_PATH         = "../dataset/test.jsonl"
TYPE_CHART_PATH        = "../dataset/type_chart_gen2.csv"
STATS_MAP_PATH         = "../temp/stats_map.json"
SAMPLE_SUBMISSION_PATH = "../dataset/sample_submission.csv"
SUBMISSION_PATH        = "submission.csv"

Path("../temp").mkdir(exist_ok=True)


# --- FEATURES ---
TEAM_SIZE = 6

STATUSES_WEIGHTS = {
    "fnt": 6,
    "slp": 2,
    "frz": 4,
    "tox": 3,
    "psn": 1,
    "par": 2,
    "brn": 2
}

BOOSTS_WEIGHTS = {
    "atk": 2,
    "def": 1,
    "spa": 2,
    "spd": 1,
    "spe": 2
}

MULTS_WEIGHTS = {
    0:      -4,
    0.25: -1.5,
    0.5:  -0.5,
    1:       0,
    2:     1.5,
    4:       3
}

EFFECT_WEIGHTS = {
    "attract":      -0.5,
    "beatup":        0.3,
    "bide":          0.4,
    "bind":         -0.4,
    "clamp":        -1.0,
    "confusion":    -0.8,
    "curse":        -0.8,
    "destinybond":   0.6,
    "disable":      -0.4,
    "encore":       -1.2,
    "endure":       -0.5,
    "firespin":     -0.4,
    "focusband":    -0.8,
    "focusenergy":  -0.4,
    "foresight":    -0.2,
    "futuresight":  -0.7,
    "leechseed":    -0.8,
    "lockon":        0.2,
    "magnitude":     0.3,
    "mimic":        -0.6,
    "mindreader":   -0.5,
    "mysteryberry": -0.3,
    "nightmare":    -1.0,
    "noeffect":      0.0,
    "perish":        0.0,
    "protect":       0.0,
    "pursuit":      -1.5,
    "quickclaw":    -0.6,
    "safeguard":     0.6,
    "spite":        -0.3,
    "substitute":    1.0,
    "trapped":      -0.9,
    "typechange":    0.3,
    "whirlpool":    -0.4,
    "wrap":         -0.4
}

ALLOWED_FEATURES = [
    # 1. Team Health & Headcount (rimosso diff_fnt perché collineare al 100% con diff_pokemon_alive)
    "diff_lst_hp_mean",
    "p1_lst_hp_mean",
    "p2_lst_hp_mean",
    "p1_fnt",
    "p2_fnt",
    "diff_pokemon_alive",
    # 2. HP Loss
    "diff_hp_pct_lost_20_30",
    "p1_hp_pct_lost_20_30",
    "p2_hp_pct_lost_20_30",
    "diff_hp_pct_lost_10_20",
    "p1_hp_pct_lost_10_20",
    "p2_hp_pct_lost_10_20",
    "diff_hp_pct_lost_0_10",
    # 3. Statuses
    "diff_lst_statuses_wmean",
    "p1_lst_statuses_wmean",
    "p2_lst_statuses_wmean",
    "diff_statuses_wmean",
    # 4. Stat Stages & Boosts
    "diff_lst_boosts_wmean",
    "p1_lst_boosts_wmean",
    "p2_lst_boosts_wmean",
    "diff_boosts_wmean",
    "p1_boosts_wmean",
    "p2_boosts_wmean",
    # 5. Type Effectiveness & Battle Effects
    "p1_mult_wmean",
    "p2_mult_wmean",
    "diff_0_mult",
    "diff_2_mult",
    "diff_mult_wmean",
    "diff_mult_per_atk",
    "diff_eff_net_mean",
    "diff_lst_effects_wmean",
    "p1_lst_effects_wmean",
    # 6. Core Defensive Base Stats
    "diff_mean_def",
    "diff_mean_hp",
    "p1_mean_def",
    "p2_mean_hp",
    # 7. Held Items (Pruned_47: rimossi p1_item_non_leftovers e p1_has_lightball_pikachu)
    "p1_item_leftovers_count",
    "p1_item_has_all_leftovers",
    "p1_item_unique_count",
    "p1_item_junk_count",
    "p1_item_status_berry_count",
    "p1_item_type_boost_count",
    "p1_item_rng_count",
    "p1_has_thickclub_marowak",
    "p2_detected_leftovers_count",
    "diff_leftovers_count",
    "diff_junk_items_count",
    # 8. Deployed Counts, Active Level & Effective HP Ratio
    "p1_active_wmean_lvl",
    "p1_deployed_count",
    "p2_deployed_count",
    "diff_deployed_count",
    "p1_rem_effective_hp_ratio",
    "p2_rem_effective_hp_ratio",
    "diff_rem_effective_hp_ratio",
    # 9. Tactical Tempo, Effective Power, Hazards & RestTalk
    "diff_moves_used",
    "diff_switches",
    "p1_eff_power_mean",
    "p2_eff_power_mean",
    "diff_eff_power_mean",
    "diff_rest_used",
    "diff_sleeptalk_used",
    "diff_spikes_active",
    "diff_phaze_used",

    "p1_hp_pct_lost_0_10", 
    "p2_hp_pct_lost_0_10",
    "diff_mean_atk", 
    "diff_mean_spa", 
    "diff_mean_spd", 
    "diff_mean_spe", 
    "diff_mean_bst",
    "p2_stat_coverage",
    "p2_lst_effects_wmean", 
    "p2_lst_boosts_wmean",
    "p1_type_diversity", 
    "p2_type_diversity", 
    "diff_type_diversity",
    "p1_type_weakness_exposure", 
    "p2_type_weakness_exposure", 
    "diff_type_weakness_exposure",
    "p1_shared_weakness_count", 
    "p2_shared_weakness_count", 
    "diff_shared_weakness_count",
    "p2_detected_rng_count",
]

ALLOWED_FEATURES = ALLOWED_FEATURES + [
    name for status in STATUSES_WEIGHTS
    for name in (f"p1_status_frac_{status}", f"p2_status_frac_{status}", f"diff_status_frac_{status}")
]

ALLOWED_FEATURES = ALLOWED_FEATURES + [
    name for boost in BOOSTS_WEIGHTS
    for name in (f"p1_{boost}_boost_mean", f"p2_{boost}_boost_mean", f"diff_{boost}_boost_mean")
]

# --- HELD ITEMS DICTIONARIES & METRIC PRIORS ---
JUNK_ITEMS = {
    "berryjuice", "berry", "noitem", "pokeball", "firestone",
    "energypowder", "aspearberry", "gengarite", "alakazite", "blackbelt"
}

STATUS_BERRIES = {
    "miracleberry", "mintberry", "przcureberry", "psncureberry",
    "bitterberry", "lumberry", "chestoberry"
}

TYPE_BOOST_ITEMS = {
    "charcoal", "magnet", "blackbelt", "pinkbow", "polkadotbow",
    "mysticwater", "hardstone", "poisonbarb", "softsand", "silverpowder",
    "blackglasses", "sharpbeak", "spelltag", "miracleseed", "twistedspoon",
    "nevermeltice", "dragonscale", "dragonfang", "metalcoat"
}

RNG_ITEMS = {
    "berserkgene", "scopelens", "quickclaw", "focusband", "brightpowder", "kingsrock"
}

HEALING_MOVES = {
    "rest", "recover", "softboiled", "milkdrink", "synthesis",
    "morningsun", "moonlight", "drainpunch", "megadrain", "gigadrain", "leechlife"
}


# --- TRAINING ---
"""
"logistic_regression":
A simple linear model that weights each feature and outputs a win probability.
It is fast and easy to interpret, but it only captures straightforward relationships.
Features are standardized first.

"lightgbm":
A gradient boosting model that builds trees one after another, each correcting the errors of the previous ones.
It is usually the most accurate on tabular data.

"xgboost":
Another gradient boosting model similar to lightgbm, but with different optimizations and regularization techniques

"catboost":
Gradient boosting with symmetric (oblivious) trees, useful as a complementary tabular model.

"ensemble":
A soft-voting combination of LightGBM, XGBoost, CatBoost, and Logistic Regression.
Each branch standardizes features within the training fold before fitting.
"""
MODEL_TYPE = "ensemble"
RANDOM_STATE = 42
K_FOLDS = 5
N_REPEATS = 3

## 1. Loading Data

In [ ]:
# --- LOAD DATA ---
def load_data(file_path: str) -> list:
    """
    Loads the whole `file_path` and stores it as a list of battles
    """

    print(f"Loading data from '{file_path}'...")
    data = []
    try:
        with open(file_path, "r") as f:
            for line in f:
                data.append(json.loads(line))
        print(f"SUCCESS: loaded {len(data)} battles.")
        
    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")

    return data


# Loading data
train_data = load_data(TRAIN_FILE_PATH)
test_data  = load_data(TEST_FILE_PATH)

### Pokemon Stats & Moves Type

In [ ]:
# --- POKEMON TYPE MAP ---
def create_stats_map(train_data: list, test_data: list, file_path: str = STATS_MAP_PATH) -> dict:
    """
    Build the type map, save it as JSON, and return it
    """

    print(f"Creating pokemon stats '{file_path}'...")
    stats_map = {}
    for data in (train_data, test_data):
        for battle in data:

            for pokemon in battle["p1_team_details"]:
                if pokemon["name"] not in stats_map:
                    stats_map[pokemon["name"]] = {"base_stats": {}, "types": []}
                stats_map[pokemon["name"]]["base_stats"] = {
                    "base_hp": pokemon["base_hp"],
                    "base_atk": pokemon["base_atk"],
                    "base_def": pokemon["base_def"],
                    "base_spa": pokemon["base_spa"],
                    "base_spd": pokemon["base_spd"],
                    "base_spe": pokemon["base_spe"],
                }
                stats_map[pokemon["name"]]["types"] = pokemon["types"]

            if battle["p2_lead_details"]["name"] not in stats_map:
                stats_map[battle["p2_lead_details"]["name"]] = {"base_stats": {}, "types": []}
            lead = battle["p2_lead_details"]
            stats_map[lead["name"]]["base_stats"] = {
                "base_hp": lead["base_hp"],
                "base_atk": lead["base_atk"],
                "base_def": lead["base_def"],
                "base_spa": lead["base_spa"],
                "base_spd": lead["base_spd"],
                "base_spe": lead["base_spe"],
            }
            stats_map[lead["name"]]["types"] = lead["types"]

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(stats_map, f, indent=4)

    print(f"SUCCESS: found {len(stats_map)} pokemon.")
    return stats_map


# --- MOVES TYPE JSON ---
def normalize_type_chart(type_chart: pd.DataFrame) -> pd.DataFrame:
    """
    Return a copy with trimmed column names and lowercase, trimmed types; reject missing types.
    """

    # Clear columns headers
    type_chart.columns = type_chart.columns.astype(str).str.strip()

    # Clear columns values
    for col in ["attacking_type", "defending_type"]:
        # If a value is Nan break the run
        if type_chart[col].isna().any():
            nan_rows = type_chart[type_chart[col].isna()].index.tolist()
            raise ValueError(f"ERROR: found NaN value in column '{col}' at rows {nan_rows}.")

        # Remove white spaces and lower
        type_chart[col] = type_chart[col].astype(str).str.strip().str.lower()

    print(f"SUCCESS: cleaned {len(type_chart)} entries.")
    return type_chart

def load_type_chart(file_path: str) -> pd.DataFrame | None:
    """
    Load and normalize a type chart CSV
    """

    print(f"Loading data from '{file_path}'...")
    try:
        raw_df = pd.read_csv(file_path)
        return normalize_type_chart(raw_df)
    
    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")


# --- MULTPLIERS --- 
def get_multiplier(move_type: str, pkm_type1: str, pkm_type2: str) -> float:
    """
    Return the move's combined effectiveness against the pokemon's types
    """

    m1 = type_chart.get((move_type, pkm_type1), 1.0) if pkm_type1 else 1.0
    m2 = type_chart.get((move_type, pkm_type2), 1.0) if pkm_type2 else 1.0
    return m1 * m2


# Getting type charts
type_chart = load_type_chart(TYPE_CHART_PATH)
df_indexed = type_chart.set_index(["attacking_type", "defending_type"])
type_chart = df_indexed["multiplier"].to_dict()
stats_map = create_stats_map(train_data, test_data)

#### Items win rate map

In [ ]:
def build_smoothed_item_wr_map(train_data: list, m: float = 50.0, prior: float = 0.50) -> dict:
    """Precomputes Bayesian-smoothed win rates per item on training data."""
    stats = {}
    for battle in train_data:
        won = battle.get("player_won", 0)
        for p in battle.get("p1_team_details", []):
            item = p.get("item") or "noitem"
            if item not in stats:
                stats[item] = [0, 0]
            stats[item][0] += won
            stats[item][1] += 1

    return {
        item: (wins + m * prior) / (total + m)
        for item, (wins, total) in stats.items()
    }

ITEM_WR_MAP = build_smoothed_item_wr_map(train_data)
DEFAULT_WR = 0.50

## 2. Extract Features (both Train and Test Features)

In [ ]:
# --- BASE STATS ---
def get_stats(turn: dict, p2_stats: dict) -> None:
    """
    Store player 2's active pokemon stats in p2_stats if found in stats_map
    """

    pkm2_name = turn["p2_pokemon_state"]["name"]
    if pkm2_name in stats_map:
        stats = stats_map[pkm2_name]["base_stats"]
        p2_stats[pkm2_name] = stats

def feature_base_stats(battle: dict, p2_stats: dict, features: dict) -> None:
    """
    Add base pokemon statistics:
    - player 1 full team absolute stats (measures team building quality)
    - player 2 seen team absolute stats (measures active counters deployed)
    - differential stats (P1 - P2: higher = P1 advantage)
    """

    p1_team = battle["p1_team_details"]
    features["p2_stat_coverage"] = len(p2_stats) / TEAM_SIZE
    features["p1_mean_hp"]  = np.mean([p["base_hp"]  for p in p1_team])
    features["p1_mean_atk"] = np.mean([p["base_atk"] for p in p1_team])
    features["p1_mean_def"] = np.mean([p["base_def"] for p in p1_team])
    features["p1_mean_spa"] = np.mean([p["base_spa"] for p in p1_team])
    features["p1_mean_spd"] = np.mean([p["base_spd"] for p in p1_team])
    features["p1_mean_spe"] = np.mean([p["base_spe"] for p in p1_team])
    features["p1_mean_bst"] = features["p1_mean_hp"] + features["p1_mean_atk"] + features["p1_mean_def"] + features["p1_mean_spa"] + features["p1_mean_spd"] + features["p1_mean_spe"]

    features["p2_mean_hp"]  = np.mean([p["base_hp"]  for p in p2_stats.values()])
    features["p2_mean_atk"] = np.mean([p["base_atk"] for p in p2_stats.values()])
    features["p2_mean_def"] = np.mean([p["base_def"] for p in p2_stats.values()])
    features["p2_mean_spa"] = np.mean([p["base_spa"] for p in p2_stats.values()])
    features["p2_mean_spd"] = np.mean([p["base_spd"] for p in p2_stats.values()])
    features["p2_mean_spe"] = np.mean([p["base_spe"] for p in p2_stats.values()])
    features["p2_mean_bst"] = features["p2_mean_hp"] + features["p2_mean_atk"] + features["p2_mean_def"] + features["p2_mean_spa"] + features["p2_mean_spd"] + features["p2_mean_spe"]

    features["diff_mean_hp"]  = features["p1_mean_hp"]  - features["p2_mean_hp"]
    features["diff_mean_atk"] = features["p1_mean_atk"] - features["p2_mean_atk"]
    features["diff_mean_def"] = features["p1_mean_def"] - features["p2_mean_def"]
    features["diff_mean_spa"] = features["p1_mean_spa"] - features["p2_mean_spa"]
    features["diff_mean_spd"] = features["p1_mean_spd"] - features["p2_mean_spd"]
    features["diff_mean_spe"] = features["p1_mean_spe"] - features["p2_mean_spe"]
    features["diff_mean_bst"] = features["p1_mean_bst"] - features["p2_mean_bst"]


def feature_team_type_profile(battle: dict, p2_stats: dict, features: dict) -> None:
    """Summarize known team type diversity and repeated defensive weaknesses."""
    p1_names = [pokemon["name"] for pokemon in battle["p1_team_details"]]
    p2_names = list(p2_stats)
    attack_types = sorted({attack_type for attack_type, _ in type_chart if attack_type != "notype"})

    def profile(pokemon_names: list[str]) -> tuple[int, float, int]:
        pokemon_types = []
        for name in pokemon_names:
            types = stats_map.get(name, {}).get("types", [])
            types = [str(value).strip().lower() for value in types if value and str(value).lower() != "notype"]
            if types:
                pokemon_types.append((types + ["notype", "notype"])[:2])

        diversity = len({pokemon_type for types in pokemon_types for pokemon_type in types if pokemon_type != "notype"})
        if not pokemon_types:
            return diversity, 0.0, 0

        weakness_exposure = 0.0
        shared_weakness_count = 0
        for attack_type in attack_types:
            weak_count = 0
            for defending_types in pokemon_types:
                multiplier = get_multiplier(attack_type, defending_types[0], defending_types[1])
                weakness_exposure += max(multiplier - 1.0, 0.0)
                weak_count += multiplier > 1.0
            shared_weakness_count += weak_count >= 2

        weakness_exposure /= len(pokemon_types)
        return diversity, weakness_exposure, shared_weakness_count

    p1_diversity, p1_exposure, p1_shared = profile(p1_names)
    p2_diversity, p2_exposure, p2_shared = profile(p2_names)
    features["p1_type_diversity"] = p1_diversity
    features["p2_type_diversity"] = p2_diversity
    features["diff_type_diversity"] = p1_diversity - p2_diversity
    features["p1_type_weakness_exposure"] = p1_exposure
    features["p2_type_weakness_exposure"] = p2_exposure
    features["diff_type_weakness_exposure"] = p2_exposure - p1_exposure
    features["p1_shared_weakness_count"] = p1_shared
    features["p2_shared_weakness_count"] = p2_shared
    features["diff_shared_weakness_count"] = p2_shared - p1_shared


# --- HP LOSS ---
def get_hp_pct_lost(turn: dict, p1_hp_map: dict, p2_hp_map: dict) -> None:
    """
    Track HP loss from one turn for each observed pokemon:
    - remember its latest HP across switches, starting from full HP
    - accumulate HP loss in its three ten-turn intervals
    Healing reduces HP loss.
    """

    name = turn["p1_pokemon_state"]["name"]
    hp = turn["p1_pokemon_state"]["hp_pct"]
    if name not in p1_hp_map:
        p1_hp_map[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}
    pokemon = p1_hp_map[name]
    pokemon["losses"][(turn["turn"] - 1) // 10] += pokemon["last_hp"] - hp
    pokemon["last_hp"] = hp

    name = turn["p2_pokemon_state"]["name"]
    hp = turn["p2_pokemon_state"]["hp_pct"]
    if name not in p2_hp_map:
        p2_hp_map[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}
    pokemon = p2_hp_map[name]
    pokemon["losses"][(turn["turn"] - 1) // 10] += pokemon["last_hp"] - hp
    pokemon["last_hp"] = hp

def feature_hp_lost(p1_hp_map: dict, p2_hp_map: dict, features: dict) -> None:
    """
    Add net HP loss in each ten-turn interval and in total, divided by TEAM_SIZE.
    Unseen pokemon contribute zero loss; healing reduces loss.
    Differentials use P2 loss - P1 loss: higher = P1 advantage.
    """

    # 1. Interval HP loss as a fraction of the full team's HP
    features["p1_hp_pct_lost_0_10"]  = sum(p["losses"][0] for p in p1_hp_map.values()) / TEAM_SIZE
    features["p1_hp_pct_lost_10_20"] = sum(p["losses"][1] for p in p1_hp_map.values()) / TEAM_SIZE
    features["p1_hp_pct_lost_20_30"] = sum(p["losses"][2] for p in p1_hp_map.values()) / TEAM_SIZE

    features["p2_hp_pct_lost_0_10"]  = sum(p["losses"][0] for p in p2_hp_map.values()) / TEAM_SIZE
    features["p2_hp_pct_lost_10_20"] = sum(p["losses"][1] for p in p2_hp_map.values()) / TEAM_SIZE
    features["p2_hp_pct_lost_20_30"] = sum(p["losses"][2] for p in p2_hp_map.values()) / TEAM_SIZE

    # 2. Total team HP loss (normalized by full team of 6)
    features["p1_tot_hp_lost"] = sum(sum(p["losses"]) for p in p1_hp_map.values()) / TEAM_SIZE
    features["p2_tot_hp_lost"] = sum(sum(p["losses"]) for p in p2_hp_map.values()) / TEAM_SIZE

    # 3. Differentials (P2 loss - P1 loss: higher = P1 advantage)
    features["diff_hp_pct_lost_0_10"]  = features["p2_hp_pct_lost_0_10"] - features["p1_hp_pct_lost_0_10"]
    features["diff_hp_pct_lost_10_20"] = features["p2_hp_pct_lost_10_20"] - features["p1_hp_pct_lost_10_20"]
    features["diff_hp_pct_lost_20_30"] = features["p2_hp_pct_lost_20_30"] - features["p1_hp_pct_lost_20_30"]
    features["diff_tot_hp_lost"]       = features["p2_tot_hp_lost"] - features["p1_tot_hp_lost"]


# --- TYPE EFFECTIVENESS ---
def get_mults(turn: dict, p1_mults: dict, p2_mults: dict) -> None:
    """
    Update both players' type effectiveness multiplier counts for one turn (damaging moves only).
    """

    mv1 = turn["p1_move_details"]
    if mv1 is not None and mv1["category"] != "status":
        mv1_type = mv1["type"]
        pkm2_name = turn["p2_pokemon_state"]["name"]
        pkm2_stats = stats_map.get(pkm2_name)
        if pkm2_stats is not None:
            pkm2_types = pkm2_stats.get("types", ["notype", "notype"])
        else:
            pkm2_types = ["notype", "notype"]
        mult1 = get_multiplier(mv1_type, pkm2_types[0], pkm2_types[1])
        p1_mults[mult1] += 1

    mv2 = turn["p2_move_details"]
    if mv2 is not None and mv2["category"] != "status":
        mv2_type = mv2["type"]
        pkm1_name = turn["p1_pokemon_state"]["name"]
        pkm1_types = stats_map[pkm1_name]["types"]
        mult2 = get_multiplier(mv2_type, pkm1_types[0], pkm1_types[1])
        p2_mults[mult2] += 1

def feature_mults(p1_mults: dict, p2_mults: dict, features: dict) -> None:
    """
    Add multiplier counts, weighted scores, and per-attack effectiveness means for both players
    """

    features["p1_0_mult"]     = p1_mults[0]
    features["p1_0.25_mult"]  = p1_mults[0.25]
    features["p1_0.5_mult"]   = p1_mults[0.5]
    features["p1_1_mult"]     = p1_mults[1]
    features["p1_2_mult"]     = p1_mults[2]
    features["p1_4_mult"]     = p1_mults[4]

    features["p1_mult_wmean"] = 0
    mul1_count = 0
    for mult, value in p1_mults.items():
        features["p1_mult_wmean"] += value * MULTS_WEIGHTS[mult]
        mul1_count += 1
    features["p1_mult_wmean"] /= mul1_count

    features["p2_0_mult"]     = p2_mults[0]
    features["p2_0.25_mult"]  = p2_mults[0.25]
    features["p2_0.5_mult"]   = p2_mults[0.5]
    features["p2_1_mult"]     = p2_mults[1]
    features["p2_2_mult"]     = p2_mults[2]
    features["p2_4_mult"]     = p2_mults[4]

    features["p2_mult_wmean"] = 0
    mul2_count = 0
    for mult, value in p2_mults.items():
        features["p2_mult_wmean"] += value * MULTS_WEIGHTS[mult]
        mul2_count += 1
    features["p2_mult_wmean"] /= mul2_count

    features["diff_0_mult"]     = features["p2_0_mult"] - features["p1_0_mult"]
    features["diff_0.25_mult"]  = features["p2_0.25_mult"] - features["p1_0.25_mult"]
    features["diff_0.5_mult"]   = features["p2_0.5_mult"] - features["p1_0.5_mult"]
    features["diff_1_mult"]     = features["p1_1_mult"] - features["p2_1_mult"]
    features["diff_2_mult"]     = features["p1_2_mult"] - features["p2_2_mult"]
    features["diff_4_mult"]     = features["p1_4_mult"] - features["p2_4_mult"]
    features["diff_mult_wmean"] = features["diff_0_mult"] + features["diff_0.25_mult"] + features["diff_0.5_mult"] + features["diff_1_mult"] + features["diff_2_mult"] + features["diff_4_mult"]
    features["diff_true_mult_wmean"] = features["p1_mult_wmean"] - features["p2_mult_wmean"]

    # Media pesata reale per singolo attacco offensivo eseguito (non collineare con p1/p2_mult_wmean)
    p1_atk_total = max(sum(p1_mults.values()), 1)
    p2_atk_total = max(sum(p2_mults.values()), 1)
    features["p1_mult_per_atk"]   = (features["p1_mult_wmean"] * mul1_count) / p1_atk_total
    features["p2_mult_per_atk"]   = (features["p2_mult_wmean"] * mul2_count) / p2_atk_total
    features["diff_mult_per_atk"] = features["p1_mult_per_atk"] - features["p2_mult_per_atk"]


# --- STATUSES (ALSO FAINT) ---
def get_statuses(turn: dict, p1_status: dict, p2_status: dict) -> None:
    """
    Accumulate active status turn counts for both players, including fainting
    """
    
    pkm1_state = turn["p1_pokemon_state"]["status"]
    if pkm1_state != "nostatus":
        if pkm1_state not in p1_status:
            p1_status[pkm1_state] = 0
        p1_status[pkm1_state] += 1

    pkm2_state = turn["p2_pokemon_state"]["status"]
    if pkm2_state != "nostatus":
        if pkm2_state not in p2_status:
            p2_status[pkm2_state] = 0
        p2_status[pkm2_state] += 1

def feature_statuses(p1_status_map: dict, p2_status_map: dict, n_turns: int, features: dict) -> None:
    """
    Add status turn totals, weighted sums, and differentials.
    Pre-initializes all known statuses to eliminate train/test schema mismatches.
    """

    features["p1_statuses_wsum"] = 0
    features["p2_statuses_wsum"] = 0

    # Pre-initialize all statuses from STATUSES_WEIGHTS
    for status, weight in STATUSES_WEIGHTS.items():
        val_sts_1 = p1_status_map.get(status, 0)
        val_sts_2 = p2_status_map.get(status, 0)
        features[f"p1_{status}"] = val_sts_1
        features[f"p2_{status}"] = val_sts_2
        features[f"diff_{status}"] = val_sts_2 - val_sts_1
        features[f"p1_status_frac_{status}"] = val_sts_1 / n_turns
        features[f"p2_status_frac_{status}"] = val_sts_2 / n_turns
        features[f"diff_status_frac_{status}"] = (val_sts_2 - val_sts_1) / n_turns
        features["p1_statuses_wsum"] += val_sts_1 * weight
        features["p2_statuses_wsum"] += val_sts_2 * weight

    features["p1_statuses_wmean"] = features["p1_statuses_wsum"] / n_turns
    features["p2_statuses_wmean"] = features["p2_statuses_wsum"] / n_turns

    # Differentials: P2 minus P1 (higher = P2 suffered more status afflictions = P1 advantage)
    features["diff_statuses_wsum"]  = features["p2_statuses_wsum"] - features["p1_statuses_wsum"]
    features["diff_statuses_wmean"] = features["p2_statuses_wmean"] - features["p1_statuses_wmean"]


# --- BOOSTS (CONSISTENT SCHEMA & DIFFERENTIALS) ---
def get_boosts(turn: dict, p1_boosts: dict, p2_boosts: dict) -> None:
    """
    Accumulate signed boost stages per stat for both active pokemon.
    Update both players' totals in place using the current observed turn
    """

    pkm1_boosts = turn["p1_pokemon_state"]["boosts"]
    for boost, value in pkm1_boosts.items():
        p1_boosts[boost] += value

    pkm2_boosts = turn["p2_pokemon_state"]["boosts"]
    for boost, value in pkm2_boosts.items():
        p2_boosts[boost] += value

def feature_boosts(p1_boosts: dict, p2_boosts: dict, n_turns: int, features: dict) -> None:
    """
    Add signed boost stage totals per stat for both players, weighted sums,
    and differentials favoring P1.
    Pre-initializes all stats to eliminate train/test schema mismatches.
    """

    features["p1_boosts_wsum"] = 0
    features["p2_boosts_wsum"] = 0

    # Pre-initialize all boosts from BOOSTS_WEIGHTS
    for boost, weight in BOOSTS_WEIGHTS.items():
        val_bst_1 = p1_boosts.get(boost, 0)
        val_bst_2 = p2_boosts.get(boost, 0)
        features[f"p1_{boost}_boost_mean"] = val_bst_1 / n_turns
        features[f"p2_{boost}_boost_mean"] = val_bst_2 / n_turns
        features[f"diff_{boost}_boost_mean"] = (val_bst_1 - val_bst_2) / n_turns
        features[f"p1_{boost}_boost"] = val_bst_1
        features[f"p2_{boost}_boost"] = val_bst_2
        features[f"diff_{boost}_boost"] = val_bst_1 - val_bst_2
        features["p1_boosts_wsum"] += val_bst_1 * weight
        features["p2_boosts_wsum"] += val_bst_2 * weight

    features["p1_boosts_wmean"] = features["p1_boosts_wsum"] / n_turns
    features["p2_boosts_wmean"] = features["p2_boosts_wsum"] / n_turns

    # Differentials (P1 - P2: higher = P1 has more positive stat stages)
    features["diff_boosts_wsum"]  = features["p1_boosts_wsum"] - features["p2_boosts_wsum"]
    features["diff_boosts_wmean"] = features["p1_boosts_wmean"] - features["p2_boosts_wmean"]


# --- EFFECTS FEATURES ---
def get_effects(turn: dict, p1_effects: dict, p2_effects: dict) -> None:
    """
    Accumulate positive and negative effect weights from one turn for both players
    """

    for eff in turn["p1_pokemon_state"]["effects"]:
        w = EFFECT_WEIGHTS.get(eff, 0.0)
        if w > 0:
            p1_effects["pos"] += w
        elif w < 0:
            p1_effects["neg"] += w

    for eff in turn["p2_pokemon_state"]["effects"]:
        w = EFFECT_WEIGHTS.get(eff, 0.0)
        if w > 0:
            p2_effects["pos"] += w
        elif w < 0:
            p2_effects["neg"] += w

def feature_effects(p1_effects: dict, p2_effects: dict, n_turns: int, features: dict) -> None:
    """
    Add weighted effect scores after all observed turns and at the final turn:
    - mean positive, negative, and net effect weights for P1 and P2
    - net weighted effect difference (P1 - P2) across the battle
    """

    features["p1_eff_pos_mean"] = p1_effects["pos"] / n_turns
    features["p1_eff_neg_mean"] = p1_effects["neg"] / n_turns
    features["p2_eff_pos_mean"] = p2_effects["pos"] / n_turns
    features["p2_eff_neg_mean"] = p2_effects["neg"] / n_turns

    features["p1_eff_net_mean"]   = features["p1_eff_pos_mean"] + features["p1_eff_neg_mean"]
    features["p2_eff_net_mean"]   = features["p2_eff_pos_mean"] + features["p2_eff_neg_mean"]

    features["diff_pos_mean"]     = features["p1_eff_pos_mean"] - features["p2_eff_pos_mean"]
    features["diff_neg_mean"]     = features["p2_eff_neg_mean"] - features["p1_eff_neg_mean"]
    features["diff_eff_net_mean"] = features["p1_eff_net_mean"] - features["p2_eff_net_mean"]


# --- TEAM SNAPSHOT AT LAST TURN ---
def get_last_seen(turn: dict, p1_last_seen: dict, p2_last_seen: dict) -> None:
    """
    Store each observed pokemon's latest HP and status stages.
    Update both players' maps in place; retain switched-out pokemon's snapshots.
    """

    pkm1 = turn["p1_pokemon_state"]
    p1_last_seen[pkm1["name"]] = {
        "hp_pct": pkm1["hp_pct"],
        "status": pkm1["status"]
    }

    pkm2 = turn["p2_pokemon_state"]
    p2_last_seen[pkm2["name"]] = {
        "hp_pct": pkm2["hp_pct"],
        "status": pkm2["status"]
    }

def feature_last_seen(last_turn: dict, p1_last_seen: dict, p2_last_seen: dict, features: dict) -> None:
    """
    Add latest-known team HP, status, boost, and effect means and P1 - P2 differences.
    Divide by six; assume unseen pokemon have full HP and no status, effects, or boosts.
    Benched pokemon retain their last observed data, which may be older.
    """

    features["p1_lst_hp_mean"] = 1.0 * (TEAM_SIZE - len(p1_last_seen))
    features["p1_lst_statuses_wmean"] = 0.0
    features["p1_lst_effects_wmean"] = 0.0
    features["p1_lst_boosts_wmean"] = 0.0
    for pkm1 in p1_last_seen.values():
        features["p1_lst_hp_mean"] += pkm1["hp_pct"]
        features["p1_lst_statuses_wmean"] += STATUSES_WEIGHTS.get(pkm1["status"], 0)
    lst_pkm1_effects = last_turn["p1_pokemon_state"]["effects"]
    for eff in lst_pkm1_effects:
        features["p1_lst_effects_wmean"] += EFFECT_WEIGHTS.get(eff, 0)
    lst_pkm1_boosts = last_turn["p1_pokemon_state"]["boosts"]
    for bst, value in lst_pkm1_boosts.items():
        features["p1_lst_boosts_wmean"] += value * BOOSTS_WEIGHTS.get(bst, 0)
    features["p1_lst_hp_mean"]        /= TEAM_SIZE
    features["p1_lst_statuses_wmean"] /= TEAM_SIZE
    features["p1_lst_effects_wmean"]  /= TEAM_SIZE
    features["p1_lst_boosts_wmean"]   /= TEAM_SIZE

    features["p2_lst_hp_mean"] = 1.0 * (TEAM_SIZE - len(p2_last_seen))
    features["p2_lst_statuses_wmean"] = 0.0
    features["p2_lst_effects_wmean"] = 0.0
    features["p2_lst_boosts_wmean"] = 0.0
    for pkm2 in p2_last_seen.values():
        features["p2_lst_hp_mean"] += pkm2["hp_pct"]
        features["p2_lst_statuses_wmean"] += STATUSES_WEIGHTS.get(pkm2["status"], 0)
    lst_pkm2_effects = last_turn["p2_pokemon_state"]["effects"]
    for eff in lst_pkm2_effects:
        features["p2_lst_effects_wmean"] += EFFECT_WEIGHTS.get(eff, 0)
    lst_pkm2_boosts = last_turn["p2_pokemon_state"]["boosts"]
    for bst, value in lst_pkm2_boosts.items():
        features["p2_lst_boosts_wmean"] += value * BOOSTS_WEIGHTS.get(bst, 0)
    features["p2_lst_hp_mean"]        /= TEAM_SIZE
    features["p2_lst_statuses_wmean"] /= TEAM_SIZE
    features["p2_lst_effects_wmean"]  /= TEAM_SIZE
    features["p2_lst_boosts_wmean"]   /= TEAM_SIZE

    features["diff_lst_hp_mean"]        = features["p1_lst_hp_mean"] - features["p2_lst_hp_mean"]
    features["diff_lst_statuses_wmean"] = features["p2_lst_statuses_wmean"] - features["p1_lst_statuses_wmean"]
    features["diff_lst_effects_wmean"]  = features["p1_lst_effects_wmean"] - features["p2_lst_effects_wmean"]
    features["diff_lst_boosts_wmean"]   = features["p1_lst_boosts_wmean"] - features["p2_lst_boosts_wmean"]


# --- HELD ITEMS EXTRACTION ---
def get_items(turn: dict, p2_item_tracker: dict) -> None:
    """
    Track observed P2 item activations turn by turn:
    - detects Leftovers recovery (approx 1/16 = 6.25% HP tick without healing moves)
    - detects item signatures appearing in turn effects
    """
    p2_state = turn.get("p2_pokemon_state", {})
    p2_mon = p2_state.get("name")
    curr_hp = p2_state.get("hp_pct", 1.0)
    p2_move = turn.get("p2_move_details")
    move_name = p2_move.get("name") if p2_move else ""

    # Check Leftovers passive regeneration between consecutive turns of the same active Pokemon
    if p2_item_tracker["prev_mon"] == p2_mon and p2_item_tracker["prev_hp"] is not None:
        hp_diff = curr_hp - p2_item_tracker["prev_hp"]
        if 0.04 <= hp_diff <= 0.08 and move_name not in HEALING_MOVES:
            p2_item_tracker["leftovers_seen"].add(p2_mon)

    # Check item names explicitly exposed in turn effects
    for eff in p2_state.get("effects", []):
        if eff in JUNK_ITEMS:
            p2_item_tracker["junk_seen"].add((p2_mon, eff))
        elif eff in RNG_ITEMS:
            p2_item_tracker["rng_seen"].add((p2_mon, eff))

    p2_item_tracker["prev_mon"] = p2_mon
    p2_item_tracker["prev_hp"] = curr_hp


def feature_items(battle: dict, p2_item_tracker: dict, features: dict) -> None:
    """
    Extract Player 1 item stats, Player 2 timeline detections, and their differentials.
    """
    p1_team = battle.get("p1_team_details", [])
    p1_items = [p.get("item") or "noitem" for p in p1_team]

    # 1. Macro team item metrics
    p1_leftovers = p1_items.count("leftovers")
    features["p1_item_leftovers_count"]   = p1_leftovers
    features["p1_item_has_all_leftovers"] = int(p1_leftovers == 6)
    features["p1_item_unique_count"]      = len(set(p1_items))
    features["p1_item_non_leftovers"]     = 6 - p1_leftovers

    # 2. Archetype groupings
    p1_junk = sum(1 for it in p1_items if it in JUNK_ITEMS)
    features["p1_item_junk_count"]         = p1_junk
    features["p1_item_status_berry_count"] = sum(1 for it in p1_items if it in STATUS_BERRIES)
    features["p1_item_type_boost_count"]   = sum(1 for it in p1_items if it in TYPE_BOOST_ITEMS)
    features["p1_item_rng_count"]          = sum(1 for it in p1_items if it in RNG_ITEMS)

    # 3. High-impact item synergies
    features["p1_has_thickclub_marowak"] = int(any(
        p.get("name") in {"marowak", "cubone"} and p.get("item") == "thickclub"
        for p in p1_team
    ))
    features["p1_has_lightball_pikachu"] = int(any(
        p.get("name") == "pikachu" and p.get("item") == "lightball"
        for p in p1_team
    ))

    # 4. Player 2 timeline detections and differentials
    p2_leftovers = len(p2_item_tracker["leftovers_seen"])
    p2_junk = len(p2_item_tracker["junk_seen"])

    features["p2_detected_leftovers_count"] = p2_leftovers
    features["p2_detected_rng_count"] = len({name for name, _ in p2_item_tracker["rng_seen"]})
    features["diff_leftovers_count"]        = p1_leftovers - p2_leftovers
    features["diff_junk_items_count"]       = p1_junk - p2_junk


# --- DEPLOYED LEVELS & EFFECTIVE HP (OUTLIER-AWARE & BASE HP x LEVEL x HP% CORRELATION) ---
def calc_eff_stat(base_val: float, level: int, is_hp: bool = False) -> float:
    """
    Compute the Gen 2 effective stat from base stat and level (assuming standard max DV=15 and max StatExp).
    Correlates base stats with Pokemon level directly.
    """

    core = ((base_val + 47) * 2 * level) / 100.0
    return core + level + 10 if is_hp else core + 5

def get_deployed_turns(turn: dict, p1_deployed_turns: dict, p2_deployed_turns: dict) -> None:
    """
    Count the number of active turns spent on the field by each deployed Pokemon.
    """

    p1_name = turn["p1_pokemon_state"]["name"]
    p1_deployed_turns[p1_name] = p1_deployed_turns.get(p1_name, 0) + 1

    p2_name = turn["p2_pokemon_state"]["name"]
    p2_deployed_turns[p2_name] = p2_deployed_turns.get(p2_name, 0) + 1

def feature_deployed_and_levels(
    battle: dict,
    p1_deployed_turns: dict,
    p2_deployed_turns: dict,
    p1_hp_map: dict,
    p2_hp_map: dict,
    features: dict
) -> None:
    """
    Add high-signal, low-collinearity features tailored for the 99.4% Lvl 100 / 0.6% Underleveled distribution:
    1. P1 & P2 deployed counts, differential deployed count, and turn-weighted active level
    2. Remaining Effective HP Ratio at Turn 30 (correlates base_hp, actual level, and latest hp_pct)
    """

    p1_team = battle["p1_team_details"]
    p1_team_map = {p["name"]: p for p in p1_team}
    n_turns = max(sum(p1_deployed_turns.values()), 1)

    deployed_pkm = [p1_team_map[name] for name in p1_deployed_turns if name in p1_team_map]

    # 1. Specific indicators for level outliers (< 100) and field presence symmetry
    features["p1_deployed_count"]              = len(deployed_pkm)
    features["p2_deployed_count"]              = len(p2_deployed_turns)
    features["diff_deployed_count"]            = features["p2_deployed_count"] - features["p1_deployed_count"]
    features["p1_underleveled_count"]          = sum(1 for p in p1_team if p["level"] < 100)
    features["p1_deployed_underleveled_count"] = sum(1 for p in deployed_pkm if p["level"] < 100)
    features["p1_team_min_lvl"]                = np.min([p["level"] for p in p1_team])
    features["p1_deployed_min_lvl"]            = np.min([p["level"] for p in deployed_pkm]) if deployed_pkm else 100.0
    features["p1_active_wmean_lvl"]            = sum(
        p1_team_map[name]["level"] * turns
        for name, turns in p1_deployed_turns.items()
        if name in p1_team_map
    ) / n_turns

    # 2. Remaining Effective HP Ratio at Turn 30 (correlates Base HP, real Level, and remaining HP%)
    p1_rem_eff_hp, p1_tot_eff_hp = 0.0, 0.0
    for p in p1_team:
        eff_hp = calc_eff_stat(p["base_hp"], p["level"], is_hp=True)
        last_hp_pct = p1_hp_map[p["name"]]["last_hp"] if p["name"] in p1_hp_map else 1.0
        p1_rem_eff_hp += eff_hp * last_hp_pct
        p1_tot_eff_hp += eff_hp
    features["p1_rem_effective_hp_ratio"] = p1_rem_eff_hp / p1_tot_eff_hp if p1_tot_eff_hp > 0 else 0.0

    # For P2: use exact level for the lead, and 100 (99.4% mode) for other seen team members
    p2_lead = battle["p2_lead_details"]
    p2_rem_eff_hp, p2_tot_eff_hp = 0.0, 0.0
    for name in p2_deployed_turns:
        if name == p2_lead["name"]:
            base_hp, lvl = p2_lead["base_hp"], p2_lead["level"]
        elif name in stats_map:
            base_hp, lvl = stats_map[name]["base_stats"]["base_hp"], 100
        else:
            base_hp, lvl = 80, 100
        eff_hp = calc_eff_stat(base_hp, lvl, is_hp=True)
        last_hp_pct = p2_hp_map[name]["last_hp"] if name in p2_hp_map else 1.0
        p2_rem_eff_hp += eff_hp * last_hp_pct
        p2_tot_eff_hp += eff_hp

    # Unseen P2 Pokemon are assumed at 100% HP with standard OU bulk (base_hp=85, lvl=100)
    unseen_p2_count = max(TEAM_SIZE - len(p2_deployed_turns), 0)
    default_p2_eff_hp = calc_eff_stat(85, 100, is_hp=True)
    p2_rem_eff_hp += unseen_p2_count * default_p2_eff_hp
    p2_tot_eff_hp += unseen_p2_count * default_p2_eff_hp
    features["p2_rem_effective_hp_ratio"] = p2_rem_eff_hp / p2_tot_eff_hp if p2_tot_eff_hp > 0 else 0.0

    features["diff_rem_effective_hp_ratio"] = (
        features["p1_rem_effective_hp_ratio"] - features["p2_rem_effective_hp_ratio"]
    )


# --- TACTICAL TEMPO, OFFENSIVE POWER, HAZARDS & REST ---
def get_tactics(turn: dict, p1_tactics: dict, p2_tactics: dict) -> None:
    """
    Track turn-by-turn tactical momentum for both players:
    - voluntary switches (excluding post-KO replacements)
    - distinct fainted Pokemon
    - effective offensive power (Base Power * STAB * Type Multiplier * Accuracy)
    - Rest / Sleep Talk usage and Spikes / Phazing (Roar/Whirlwind)
    """
    for side, opp_side, tracker, opp_tracker in (
        ("p1", "p2", p1_tactics, p2_tactics),
        ("p2", "p1", p2_tactics, p1_tactics),
    ):
        state = turn[f"{side}_pokemon_state"]
        opp_state = turn[f"{opp_side}_pokemon_state"]
        mv = turn[f"{side}_move_details"]
        name = state["name"]
        is_fainted = (state["status"] == "fnt" or state["hp_pct"] == 0.0)

        # Conta solo i cambi volontari (esclude gli ingressi forzati dopo un KO al turno precedente)
        if tracker["prev_mon"] is not None and tracker["prev_mon"] != name:
            if not tracker.get("prev_fainted", False):
                tracker["switches"] += 1
        tracker["prev_mon"] = name
        tracker["prev_fainted"] = is_fainted

        # Track distinct fainted Pokemon
        if is_fainted:
            tracker["fainted_mons"].add(name)

        if mv is not None:
            tracker["moves_used"] += 1
            mv_name = mv["name"]
            mv_type = mv["type"]

            # Azioni di stato valide solo se il Pokemon non è andato KO prima di eseguirle
            if not is_fainted:
                if mv_name == "rest":
                    tracker["rest_used"] += 1
                elif mv_name == "sleeptalk":
                    tracker["sleeptalk_used"] += 1
                elif mv_name == "spikes":
                    tracker["spikes_active"] = 1
                elif mv_name == "rapidspin":
                    opp_tracker["spikes_active"] = 0
                elif mv_name in {"roar", "whirlwind"}:
                    tracker["phaze_used"] += 1

            # Effective offensive power for damaging moves
            if mv["category"] != "status" and mv["base_power"] > 0:
                attacker_types = stats_map.get(name, {}).get("types", ["notype", "notype"])
                defender_types = stats_map.get(opp_state["name"], {}).get("types", ["notype", "notype"])
                stab = 1.5 if mv_type in attacker_types else 1.0
                mult = get_multiplier(mv_type, defender_types[0], defender_types[1])
                acc = mv.get("accuracy") or 1.0
                tracker["power_sum"] += mv["base_power"] * stab * mult * acc


def feature_tactics(p1_tactics: dict, p2_tactics: dict, n_turns: int, features: dict) -> None:
    """Compute tactical differentials from accumulated timeline trackers."""
    # 1. True distinct fainted counts & alive differential
    p1_ko = len(p1_tactics["fainted_mons"])
    p2_ko = len(p2_tactics["fainted_mons"])
    features["p1_distinct_ko"]     = p1_ko
    features["p2_distinct_ko"]     = p2_ko
    features["diff_pokemon_alive"] = p2_ko - p1_ko

    # 2. Action tempo & switching
    features["p1_moves_used"]   = p1_tactics["moves_used"] / n_turns
    features["p2_moves_used"]   = p2_tactics["moves_used"] / n_turns
    features["diff_moves_used"] = (p1_tactics["moves_used"] - p2_tactics["moves_used"]) / n_turns
    features["p1_switches"]     = p1_tactics["switches"] / n_turns
    features["p2_switches"]     = p2_tactics["switches"] / n_turns
    features["diff_switches"]   = (p1_tactics["switches"] - p2_tactics["switches"]) / n_turns

    # 3. Effective offensive power per turn
    features["p1_eff_power_mean"]   = p1_tactics["power_sum"] / n_turns
    features["p2_eff_power_mean"]   = p2_tactics["power_sum"] / n_turns
    features["diff_eff_power_mean"] = features["p1_eff_power_mean"] - features["p2_eff_power_mean"]

    # 4. Gen 2 Meta: RestTalk & Spikes/Phazing
    features["p1_rest_used"]        = p1_tactics["rest_used"]
    features["p2_rest_used"]        = p2_tactics["rest_used"]
    features["diff_rest_used"]      = p1_tactics["rest_used"] - p2_tactics["rest_used"]
    features["p1_sleeptalk_used"]   = p1_tactics["sleeptalk_used"]
    features["p2_sleeptalk_used"]   = p2_tactics["sleeptalk_used"]
    features["diff_sleeptalk_used"] = p1_tactics["sleeptalk_used"] - p2_tactics["sleeptalk_used"]
    features["p1_spikes_active"]    = p1_tactics["spikes_active"]
    features["p2_spikes_active"]    = p2_tactics["spikes_active"]
    features["diff_spikes_active"]  = p1_tactics["spikes_active"] - p2_tactics["spikes_active"]
    features["p1_phaze_used"]       = p1_tactics["phaze_used"]
    features["p2_phaze_used"]       = p2_tactics["phaze_used"]
    features["diff_phaze_used"]     = p1_tactics["phaze_used"] - p2_tactics["phaze_used"]

In [ ]:
# --- EXTRACT FEATURES ---
def extract_features(data: list) -> pd.DataFrame:
    """
    Extract comprehensive features combining absolute scale and differentials:
    1. base stats (P1 absolute, P2 absolute, diff P1 - P2)
    2. HP loss in intervals and total (P1, P2, and differentials)
    3. type effectiveness multiplier counts and differentials
    4. statuses (pre-initialized, consistent schema, and differentials)
    5. boosts (P1, P2, and differentials)
    6. battle effects (P1, P2, and differentials)
    7. last-turn snapshot at Turn 30 (P1, P2, and differentials)
    8. held items (P1 known items, P2 timeline detections, and differentials)
    9. team type profile and defensive matchup summaries
    10. deployed Pokemon levels, symmetry counts & effective HP ratios
    11. tactical tempo, effective offensive power, hazards, phazing & RestTalk
    """

    print("Extracting features...")
    feature_list = []

    for battle in data:
        features = {"battle_id" : battle["battle_id"]}

        # Keep the identifier and target for the training and submission cells
        if "player_won" in battle:
            features["player_won"] = battle["player_won"]


        # Initializations
        p1_hp_map, p2_hp_map = {}, {}
        p1_mults = {mult: 0 for mult in MULTS_WEIGHTS}
        p2_mults = {mult: 0 for mult in MULTS_WEIGHTS}
        p1_status, p2_status = {}, {}
        p1_boosts = {boost: 0 for boost in BOOSTS_WEIGHTS}
        p2_boosts = {boost: 0 for boost in BOOSTS_WEIGHTS}
        p1_effects = {"pos": 0.0, "neg": 0.0}
        p2_effects = {"pos": 0.0, "neg": 0.0}
        p1_last_seen, p2_last_seen = {}, {}
        p2_stats = {}
        p2_item_tracker = {
            "leftovers_seen": set(),
            "junk_seen": set(),
            "rng_seen": set(),
            "prev_mon": None,
            "prev_hp": None,
        }
        p1_deployed_turns, p2_deployed_turns = {}, {}
        p1_tactics = {
            "prev_mon": None, "prev_fainted": False, "switches": 0, "fainted_mons": set(), "moves_used": 0,
            "rest_used": 0, "sleeptalk_used": 0, "spikes_active": 0, "phaze_used": 0, "power_sum": 0.0
        }
        p2_tactics = {
            "prev_mon": None, "prev_fainted": False, "switches": 0, "fainted_mons": set(), "moves_used": 0,
            "rest_used": 0, "sleeptalk_used": 0, "spikes_active": 0, "phaze_used": 0, "power_sum": 0.0
        }

        timeline = battle["battle_timeline"]
        for turn in timeline:
            get_stats(turn, p2_stats)
            get_hp_pct_lost(turn, p1_hp_map, p2_hp_map)
            get_mults(turn, p1_mults, p2_mults)
            get_statuses(turn, p1_status, p2_status)
            get_boosts(turn, p1_boosts, p2_boosts)
            get_effects(turn, p1_effects, p2_effects)
            get_last_seen(turn, p1_last_seen, p2_last_seen)
            get_items(turn, p2_item_tracker)
            get_deployed_turns(turn, p1_deployed_turns, p2_deployed_turns)
            get_tactics(turn, p1_tactics, p2_tactics)

        last_turn = timeline[-1]
        # 1. Base stats
        feature_base_stats(battle, p2_stats, features)
        feature_team_type_profile(battle, p2_stats, features)
        # 2. HP loss in intervals, totals & differentials
        feature_hp_lost(p1_hp_map, p2_hp_map, features)
        # 3. Type effectiveness multiplier counts & differentials
        feature_mults(p1_mults, p2_mults, features)
        # 4. Statuses & differentials
        feature_statuses(p1_status, p2_status, len(timeline), features)
        # 5. Boosts & differentials
        feature_boosts(p1_boosts, p2_boosts, len(timeline), features)
        # 6. Effects & differentials
        feature_effects(p1_effects, p2_effects, len(timeline), features)
        # 7. Last seen & differentials
        feature_last_seen(last_turn, p1_last_seen, p2_last_seen, features)
        # 8. Items
        feature_items(battle, p2_item_tracker, features)
        # 9. Deployed Pokemon levels & effective stats (Base Stats x Level)
        feature_deployed_and_levels(battle, p1_deployed_turns, p2_deployed_turns, p1_hp_map, p2_hp_map, features)
        # 10. Tactical tempo, offensive power, hazards & RestTalk
        feature_tactics(p1_tactics, p2_tactics, len(timeline), features)

        feature_list.append(features)

    print(f"SUCCESS: extracted {len(feature_list[0])} features.")
    return pd.DataFrame(feature_list).fillna(0)


# Extracting training and test features
train_df = extract_features(train_data)
print(f"Training features: {train_df.shape}.")
display(train_df.head())

test_df = extract_features(test_data)
print(f"Test features: {test_df.shape}.")
# display(test_df.head())

## 3. Training

### Models

In [ ]:
# --- BUILD MODEL ---
def make_lr(random_state: int): # -> Pipeline:
    """
    Build an instance of logistic_regression with seed `random_state`
    """
    
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=1.0, random_state=random_state)
    )

def make_lgbm(random_state: int) -> LGBMClassifier:
    """
    Build an instance of `lightgbm` with seed `random_state`
    """
        
    return make_pipeline(
        StandardScaler(),
        LGBMClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        verbose=-1,
        random_state=random_state
        )
    )

def make_xgb(random_state: int) -> XGBClassifier:
    """
    Build an instance of `xgboost` with seed `random_state`
    """
    return make_pipeline(
        StandardScaler(),
        XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=-1
        )
    )

def make_catboost(random_state: int):
    """Build CatBoost using symmetric (oblivious) trees."""
    return make_pipeline(
        StandardScaler(),
        CatBoostClassifier(
            iterations=150,
            depth=5,
            learning_rate=0.04,
            loss_function="Logloss",
            verbose=False,
            allow_writing_files=False,
            random_seed=random_state,
            thread_count=1
        )
    )

def build_model(model_type: str = MODEL_TYPE, random_state: int = RANDOM_STATE): # -> BaseEstimator:
    """
    Build an instance of `model_type` with seed `random_state`
    """

    match model_type:
        case "logistic_regression":
            return make_lr(random_state)

        case "lightgbm":
            return make_lgbm(random_state)

        case "xgboost":
            return make_xgb(random_state)

        case "catboost":
            return make_catboost(random_state)

        case "ensemble":
            return VotingClassifier(
                estimators=[
                    ("lgb", make_lgbm(random_state)),
                    ("xgb", make_xgb(random_state)),
                    ("cat", make_catboost(random_state)),
                    ("lr", make_lr(random_state)),
                ],
                voting="soft",
                weights=[2.0, 2.0, 1.0, 2.5],
            )

        case _:
            raise ValueError(f"ERROR: Unavailable model: {model_type}.")

### Perfomance Metrics

In [ ]:
# --- METRICS ---
def compute_metrics(y_true: pd.Series | np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> dict:
    """
    Return accuracy, precision, recall, F1, and ROC-AUC for binary predictions.
    y_true and y_pred contain 0/1 labels.
    y_prob contains probabilities of class 1
    """

    return {
        # (TP + TN) / total
        "accuracy": accuracy_score(y_true, y_pred),
        # TP / (TP + FP)
        "precision": precision_score(y_true, y_pred, zero_division=0),
        # TP / (TP + FN)
        "recall": recall_score(y_true, y_pred, zero_division=0),
        # 2 * (precision * recall) / (precision + recall) = 2 * TP / (2 * TP + FP + FN)
        "f1": f1_score(y_true, y_pred, zero_division=0),
        # Area Under the Receiver Operating Characteristic Curve
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

### Training

In [ ]:

# --- TRAINING ---
def train_and_evaluate_cv(X: pd.DataFrame, y: pd.Series, X_test: pd.DataFrame, model_type: str = MODEL_TYPE, k: int = K_FOLDS, n_repeats: int = N_REPEATS, random_state: int = RANDOM_STATE):

    """
    Repeated Stratified K-Fold Cross-Validation:
    1. Computes training and validation metrics for the ensemble
       and its individual estimators.
    2. Computes OOF probabilities, averaged across repeats,
       for the ensemble and each individual estimator.
    3. Averages the ensemble test-set probabilities over all folds.
    """

    rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=random_state)

    n_total = k * n_repeats

    # Model names
    model_names = ["ensemble", "lgb", "xgb", "cat", "lr"]

    display_names = {
        "ensemble": "Ensemble",
        "lgb": "LightGBM",
        "xgb": "XGBoost",
        "cat": "CatBoost",
        "lr": "Logistic Regression"
    }

    # Initialize OOF and test probabilities
    oof_probs = {name: np.zeros(len(X)) for name in model_names}
    test_probs = np.zeros(len(X_test))

    # Initialize metric storage
    metrics_by_model = {name: {"train": [], "val": []} for name in model_names}

    print(f"Starting {n_repeats}x Stratified {k}-Fold CV with model: {model_type}.")

    # Cross-validation
    for i, (train_idx, val_idx) in enumerate(rskf.split(X, y)):

        repeat, fold = divmod(i, k)

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # Build and train ensemble
        ensemble = build_model(model_type, random_state=random_state)
        ensemble.fit(X_train, y_train)

        # Retrieve fitted individual estimators
        fitted_models = {"ensemble": ensemble, **dict(ensemble.named_estimators_)}

        # Evaluate ensemble and individual models
        for name, estimator in fitted_models.items():

            # Training predictions
            train_prob = estimator.predict_proba(X_train)[:, 1]
            train_pred = (train_prob >= 0.5).astype(int)

            train_metrics = compute_metrics(
                y_train, train_pred, train_prob
            )

            metrics_by_model[name]["train"].append(train_metrics)

            # Validation predictions
            val_prob = estimator.predict_proba(X_val)[:, 1]
            val_pred = (val_prob >= 0.5).astype(int)

            val_metrics = compute_metrics(
                y_val, val_pred, val_prob
            )

            metrics_by_model[name]["val"].append(val_metrics)

            # Accumulate OOF probabilities across repeats
            oof_probs[name][val_idx] += val_prob / n_repeats

        # Print ensemble validation metrics for current fold
        ensemble_metrics = metrics_by_model["ensemble"]["val"][-1]
        metrics_text = "".join(f"{name}: {value:.4f} | " for name, value in ensemble_metrics.items())
        print(f"Repeat {repeat+1}/{n_repeats} | Fold {fold+1}/{k} -> {metrics_text}")

        # Aggregate ensemble test probabilities
        test_probs += ensemble.predict_proba(X_test)[:, 1] / n_total

    print_val_train_metrics(n_repeats, k, n_total, model_names, metrics_by_model, oof_probs, y, display_names)
    print_oof_metrics(n_repeats, k, n_total, model_names, metrics_by_model, oof_probs, y, display_names)
    print_overfitting_diagnostics(model_names, metrics_by_model, display_names)

    # Final ensemble test predictions
    test_predictions = (test_probs >= 0.5).astype(int)

    return test_predictions, test_probs


def print_val_train_metrics(n_repeats: int, k: int, n_total: int, model_names: list, metrics_by_model: dict, oof_probs: dict, y: pd.Series, display_names: dict) -> None:
    print(f"\nAggregated cv results ({n_repeats}x{k} folds = {n_total} evaluations):")
    for dataset in ["val", "train"]:
        print(f"\n{dataset.upper()} METRICS")
        print(f"----------------------------------------------------------------------")
        for name in model_names:
            fold_metrics = metrics_by_model[name][dataset]

            mean_acc = np.mean([m["accuracy"] for m in fold_metrics])
            std_acc  = np.std([m["accuracy"] for m in fold_metrics])
            mean_f1  = np.mean([m["f1"] for m in fold_metrics])
            mean_auc = np.mean([m["roc_auc"] for m in fold_metrics])

            print(
                f"{display_names[name]:<22} | "
                f"Accuracy: {mean_acc:.4f} (+/- {std_acc:.4f}) | "
                f"F1: {mean_f1:.4f} | "
                f"ROC-AUC: {mean_auc:.4f}"
            )

def print_oof_metrics(n_repeats: int, k: int, n_total: int, model_names: list, metrics_by_model: dict, oof_probs: dict, y: pd.Series, display_names: dict) -> None:
    print("\nOVERALL OOF METRICS")
    print(f"----------------------------------------------------------------------")
    for name in model_names:
        oof_pred = (oof_probs[name] >= 0.5).astype(int)
        overall_oof_acc = accuracy_score(y, oof_pred)
        overall_oof_auc = roc_auc_score(y, oof_probs[name])

        print(
            f"{display_names[name]:<22} | "
            f"Accuracy: {overall_oof_acc:.4f} | "
            f"ROC-AUC: {overall_oof_auc:.4f}"
        )

def print_overfitting_diagnostics(model_names: list, metrics_by_model: dict, display_names: dict) -> None:
    print("\nOVERFITTING / UNDERFITTING DIAGNOSTICS")
    print("=" * 100)
    for name in model_names:

        train_metrics = metrics_by_model[name]["train"]
        val_metrics = metrics_by_model[name]["val"]

        # Mean training metrics
        train_acc = np.mean([m["accuracy"] for m in train_metrics])
        train_f1 = np.mean([m["f1"] for m in train_metrics])
        train_auc = np.mean([m["roc_auc"] for m in train_metrics])

        # Mean validation metrics
        val_acc = np.mean([m["accuracy"] for m in val_metrics])
        val_f1 = np.mean([m["f1"] for m in val_metrics])
        val_auc = np.mean([m["roc_auc"] for m in val_metrics])

        # Generalization gaps
        acc_gap = train_acc - val_acc
        f1_gap = train_f1 - val_f1
        auc_gap = train_auc - val_auc

        # Heuristic based on the gaps
        if max(acc_gap, f1_gap, auc_gap) >= 0.10:    status = "Possible overfitting"
        elif max(acc_gap, f1_gap, auc_gap) >= 0.05:  status = "Mild overfitting"
        elif min(acc_gap, f1_gap, auc_gap) < -0.05:  status = "Validation exceeds training"
        else:                                        status = "No strong generalization gap"

        print(f"\n{display_names[name]}")
        print(f"----------------------------------------------------------------------")
        print(f"Accuracy | Train: {train_acc:.4f} | Validation: {val_acc:.4f} | Gap: {acc_gap:+.4f}")
        print(f"F1       | Train: {train_f1:.4f}  | Validation: {val_f1:.4f} | Gap: {f1_gap:+.4f}")
        print(f"ROC-AUC  | Train: {train_auc:.4f} | Validation: {val_auc:.4f} | Gap: {auc_gap:+.4f}")
        print(f"Status: {status}")

# Features excluding the ID and target columns
feature_cols = [
    c for c in train_df.columns
    if c not in ["battle_id", "player_won"] and c in ALLOWED_FEATURES
]
missing_allowed_features = [
    feature for feature in ALLOWED_FEATURES if feature not in train_df.columns
]

# Add any columns from feature_cols missing in test_df, setting every row to 0.0
for col in feature_cols:
    if col not in test_df.columns:
        test_df[col] = 0.0

print(f"allowed features: {len(ALLOWED_FEATURES)}")
print(f"selected features: {len(feature_cols)}")
print(f"unavailable allowed features: {missing_allowed_features}")

X_train = train_df[feature_cols]
y_train = train_df["player_won"]
X_test = test_df[feature_cols]

# Training, K-Fold Validation, and Ensemble Prediction
test_predictions, test_probabilities = train_and_evaluate_cv(
    X=X_train,
    y=y_train,
    X_test=X_test,
    model_type=MODEL_TYPE,
    k=K_FOLDS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

## 4. Submission File

In [ ]:
def predict_and_save(predictions: np.ndarray, test_battle_ids: pd.Series, output_path: str = SUBMISSION_PATH, sample_path: str = SAMPLE_SUBMISSION_PATH) -> None:
    """
    Format and save the submission file using the required template
    """
    
    print(f"Saving predictions to '{output_path}'...")
    sample_submission = pd.read_csv(sample_path)
    expected_columns = ["battle_id", "player_won"]
    if list(sample_submission.columns) != expected_columns:
        raise ValueError(f"ERROR: expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

    predictions_by_id = pd.Series(predictions, index=test_battle_ids)
    sample_submission["player_won"] = sample_submission["battle_id"].map(predictions_by_id)

    if sample_submission["player_won"].isna().any():
        raise ValueError(f"ERROR: some battle IDs in the '{sample_path}' have no matching prediction.")

    sample_submission["player_won"] = sample_submission["player_won"].astype(int)
    sample_submission.to_csv(output_path, index=False)

    print(f"SUCCESS: file '{output_path}' created.")
    print(sample_submission.head(10))
    print(f"Prediction distribution (0 vs 1): {dict(sample_submission['player_won'].value_counts())}")


# Submission file
predict_and_save(test_predictions, test_df["battle_id"], SUBMISSION_PATH, SAMPLE_SUBMISSION_PATH)

## Debug

In [ ]:
# --- BATTLE EXAMPLE ---
def print_battle(data, battle_idx=0):
    """
    Prints the whole battle at index `battle_idx`
    """
    print(f"\n-----------------print_battle()-----------------")

    battle = data[battle_idx]
    output_path = f"../temp/battle_{battle_idx}.json"
    with open(output_path, "w") as f:
        json.dump(battle, f, indent=4)

    print(f"SUCCESS: battle saved to {output_path}.")


def diagnose_type_chart(type_chart, stats_map, train_data):
    """Check that type names in the chart match the training data."""
    print(f"\n-----------------diagnose_type_chart()-----------------")

    chart_attack = {a for a, d in type_chart.keys()}
    chart_defend = {d for a, d in type_chart.keys()}

    pokemon_types = {pokemon_type for pokemon in stats_map.values() for pokemon_type in pokemon["types"] if pokemon_type}
    move_types, move_categories = set(), set()
    for battle in train_data:
        for turn in battle["battle_timeline"]:
            for key in ("p1_move_details", "p2_move_details"):
                move = turn[key]
                if move is not None:
                    move_types.add(move["type"])
                    move_categories.add(move["category"])

    print("Tipi in tabella (attacco):", sorted(map(repr, chart_attack)))
    print("Tipi in tabella (difesa) :", sorted(map(repr, chart_defend)))
    print("Tipi Pokemon nei dati    :", sorted(map(repr, pokemon_types)))
    print("Tipi mosse nei dati      :", sorted(map(repr, move_types)))
    print("Categorie mosse          :", sorted(map(repr, move_categories)))
    print("\nTipi di mosse NON presenti in tabella  :", move_types - chart_attack)
    print("Tipi di Pokemon NON presenti in tabella:", pokemon_types - chart_defend)

    print("\nProva su coppie (tipo mossa, tipo Pokemon):")
    for move_type in sorted(move_types)[:3]:
        for pokemon_type in sorted(pokemon_types)[:3]:
            value = type_chart.get((move_type, pokemon_type), "1.00")
            print(f"  {move_type!r:12} vs {pokemon_type!r:12} -> {value}")


def diagnose_fnt_features(train_df):
    """Summarize the aggregate and differential alive/fnt features."""
    print(f"\n-----------------diagnose_fnt_features()-----------------")
    cols = [c for c in ["diff_pokemon_alive", "p1_fainted_count", "p2_fainted_count", "p1_fnt", "p2_fnt"] if c in train_df.columns]
    print(train_df[cols].describe())
    print(train_df.groupby("player_won")[cols].mean())


def diagnose_ko_records(train_data):
    """Inspect how fainted Pokemon are recorded in battle timelines."""
    print(f"\n-----------------diagnose_ko_records()-----------------")

    rows = []
    for battle in train_data:
        timeline = battle["battle_timeline"]
        row = {}
        for player in (1, 2):
            states = [turn[f"p{player}_pokemon_state"] for turn in timeline]
            row[f"p{player}_turns_status_fnt"] = sum(state["status"] == "fnt" for state in states)
            row[f"p{player}_turns_hp0"] = sum(state["hp_pct"] == 0 for state in states)
            row[f"p{player}_distinct_names"] = len({state["name"] for state in states})
            row[f"p{player}_distinct_ko"] = len({state["name"] for state in states if state["hp_pct"] == 0 or state["status"] == "fnt"})
        rows.append(row)
    print(pd.DataFrame(rows).mean().round(3).to_string())


def diagnose_species_stats(train_data, test_data):
    """Check consistency of base stats and levels across the datasets."""
    from collections import Counter, defaultdict

    print(f"\n-----------------diagnose_species_stats()-----------------")

    base_stats = ["base_hp", "base_atk", "base_def", "base_spa", "base_spd", "base_spe"]
    stats_seen = defaultdict(set)
    levels_seen = defaultdict(Counter)

    for data in (train_data, test_data):
        for battle in data:
            for pokemon in battle["p1_team_details"] + [battle["p2_lead_details"]]:
                stats_seen[pokemon["name"]].add(tuple(pokemon[key] for key in base_stats))
                levels_seen[pokemon["name"]][pokemon["level"]] += 1

    print("Specie viste:", len(stats_seen))
    inconsistent = {name for name, stats in stats_seen.items() if len(stats) > 1}
    print("Specie con piu' set di statistiche base:", len(inconsistent), list(inconsistent)[:5])

    total_levels = Counter()
    for counts in levels_seen.values():
        total_levels.update(counts)
    print("Distribuzione livelli:", sorted(total_levels.items()))

    seen_p2 = {turn["p2_pokemon_state"]["name"] for battle in train_data + test_data for turn in battle["battle_timeline"]}
    print("Specie di P2 senza statistiche:", seen_p2 - set(stats_seen))


def diagnose_distinct_fnt(train_data):
    """Count distinct fainted Pokemon for P1 and P2."""
    print(f"\n-----------------diagnose_distinct_fnt()-----------------")

    p1_fnt_names_list, p2_fnt_names_list = [], []
    for battle in train_data:
        p1_fnt_names = set()
        p2_fnt_names = set()

        for turn in battle["battle_timeline"]:
            state1 = turn["p1_pokemon_state"]
            state2 = turn["p2_pokemon_state"]

            if state1["status"] == "fnt" or state1["hp_pct"] == 0:
                p1_fnt_names.add(state1["name"])
            if state2["status"] == "fnt" or state2["hp_pct"] == 0:
                p2_fnt_names.add(state2["name"])

        p1_fnt_names_list.append(len(p1_fnt_names))
        p2_fnt_names_list.append(len(p2_fnt_names))

    print("Media Pokemon distinti con status=='fnt' o hp==0 -> " f"P1: {np.mean(p1_fnt_names_list):.4f} | " f"P2: {np.mean(p2_fnt_names_list):.4f}")


def print_features_correlation(X_train, y_train):
    """
    Analizza la correlazione lineare tra ogni feature e la vittoria di P1 (player_won).
    Separa le feature che depongono a favore (correlazione positiva)
    da quelle che depongono a sfavore (correlazione negativa).
    """
    print(f"\n-----------------print_features_correlation()-----------------")
    corr = X_train.corrwith(y_train)

    nan_cols = corr[corr.isna()].index.tolist()
    if nan_cols:
        print(f"Feature con correlazione NaN (costanti): {len(nan_cols)} -> {nan_cols}")

    valid_corr = corr.dropna()
    sorted_corr = valid_corr.loc[valid_corr.abs().sort_values(ascending=False).index]
    formatted_corr = sorted_corr.map(lambda value: f"{value:+.6f}")

    print(f"\n=== TOTALE FEATURE DISPONIBILI: {len(valid_corr)} ===")
    print("\n=== FEATURE ORDINATE PER VALORE ASSOLUTO DELLA CORRELAZIONE ===")
    print(formatted_corr.to_string())


if DEBUG:
    print_battle(train_data)
    diagnose_type_chart(type_chart, stats_map, train_data)
    diagnose_fnt_features(train_df)
    diagnose_ko_records(train_data)
    diagnose_species_stats(train_data, test_data)
    diagnose_distinct_fnt(train_data)
    print_features_correlation(X_train, y_train)


In [ ]:
# --- EXTRACT HELD ITEMS SUMMARY WITH WIN RATE ---
def get_items_summary(train_data: list, test_data: list) -> dict:
    """
    Extracts item frequencies, win rate (calculated on train_data), and the set of 
    Pokémon holding each item across all battles in train and test datasets.
    """
    raw_item_map = {}

    def init_item(name: str):
        if name not in raw_item_map:
            raw_item_map[name] = {
                "total_count": 0,
                "train_count": 0,
                "train_wins": 0,
                "pokemon_set": set()
            }

    # 1. Process train_data (where ground truth player_won is known)
    for battle in train_data:
        won = battle.get("player_won", 0) == 1
        for pokemon in battle.get("p1_team_details", []):
            item_name = pokemon.get("item") or "noitem"
            pkm_name = pokemon.get("name")

            init_item(item_name)
            raw_item_map[item_name]["total_count"] += 1
            raw_item_map[item_name]["train_count"] += 1
            if won:
                raw_item_map[item_name]["train_wins"] += 1
            if pkm_name:
                raw_item_map[item_name]["pokemon_set"].add(pkm_name)

    # 2. Process test_data (accumulate total counts and seen Pokémon)
    for battle in test_data:
        for pokemon in battle.get("p1_team_details", []):
            item_name = pokemon.get("item") or "noitem"
            pkm_name = pokemon.get("name")

            init_item(item_name)
            raw_item_map[item_name]["total_count"] += 1
            if pkm_name:
                raw_item_map[item_name]["pokemon_set"].add(pkm_name)

    # 3. Compute win rates and structure dictionary
    items_summary = {}
    sorted_items = sorted(
        raw_item_map.items(), 
        key=lambda x: x[1]["total_count"], 
        reverse=True
    )

    for item, info in sorted_items:
        train_n = info["train_count"]
        win_rate = round(info["train_wins"] / train_n, 4) if train_n > 0 else 0.0

        items_summary[item] = {
            "count": info["total_count"],
            "win_rate": win_rate,
            "pokemon_list": sorted(info["pokemon_set"])
        }

    return items_summary


# Execute extraction
items_summary = get_items_summary(train_data, test_data)

# Display summary inspection
print(f"Total unique items found: {len(items_summary)}\n")
print(f"{'Item':<16} | {'Total Count':<11} | {'Win Rate':<10} | {'Pokémon Count':<13} | Sample Pokémon")
print("-" * 80)
for item, info in items_summary.items():
    sample_pokemon = ", ".join(info["pokemon_list"][:4])
    ellipsis_str = "..." if len(info["pokemon_list"]) > 4 else ""
    print(f"{item:<16} | {info['count']:<11} | {info['win_rate'] * 100:>6.2f}%   | {len(info['pokemon_list']):<13} | {sample_pokemon}{ellipsis_str}")